## **Business Focused Approach : How to Handle Imbalanced Data in Marketing**

### **Business Analytics in Telemarketing: Cost-Sensitive Analysis of Bank Campaigns Using Machine Learning**


### **Bank Telemarketing Campaign Analysis**

#### **Problem Statement**
Using a bank telemarketing dataset(41188 customers, with 11.3% subscription rate). Will explore different approaches to handle class imbalance through a business lens. This notebook shows how to choose the right technique based on your specific business context and costs.

In [28]:
## Import commonly used libraries
import pandas as pd 
import numpy as np  
import sidetable
import sklearn
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from glob import glob
from itertools import combinations
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
## Dispaly Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import warnings
warnings.filterwarnings('ignore')

In [3]:
## Read the data
files = []
data_path = Path.cwd().parent.joinpath('data', 'raw')
for file in data_path.glob('*'):
    files.append(file.name)

print(files, end=" ")

['.gitkeep', 'bank-additional-full.csv', 'bank-additional-names.txt', 'bank-full.csv', 'info.txt'] 

In [4]:
tele_df = pd.read_csv(data_path.joinpath('bank-additional-full.csv'), sep=';')
tele_df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [5]:
tele_df.shape

(41188, 21)

In [6]:
(tele_df['y'].value_counts(normalize=True)*100).round(1)

y
no     88.7
yes    11.3
Name: proportion, dtype: float64

__Bank Telemarketing Dataset Analysis :__

In [34]:
# ==================================================
# PART 1: LOAD BANK TELEMARKETING DATASET
# ==================================================

# Title: Bank Marketing (with social/economic context)
# Source: https://archive.ics.uci.edu/ml/datasets/Bank+Marketing
# Citation : Moro, P. Cortez and P. Rita. A Data-Driven Approach to Predict the Success of Bank Telemarketing. Decision Support Systems (2014)

def load_bank_telemarketing_data():
    """
    Load and preprocess the bank telemarketing dataset for term deposit subscription prediction.

    Returns:
    X, y, feature_names for modeling and analysis.
    """
    # Load the dataset
    tele_df = pd.read_csv(data_path.joinpath('bank-additional-full.csv'), sep=';')
    
    print(f"Dataset shape: {tele_df.shape}")

    # Prepare features and target variable
    X_df = tele_df.drop('y', axis='columns')
    y_series = tele_df['y']

    # convert target variable to binary
    y = (y_series == 'yes').astype(int)

    print("Target Distribution:")
    print(f"No Subscription (0) : {(y == 0).mean() * 100:.1f}%")
    print(f"Subscription (1): {(y == 1).mean() * 100:.1f}%")

    # Handle categorical variables with Label encoding
    X_processed = X_df.copy()
    cat_columns = X_processed.select_dtypes(include=['object']).columns

    for col in cat_columns:
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_df[col])

    X = X_processed.values
    feature_names = X_processed.columns.tolist()
    return X, y, feature_names

# Load the bank telemarketing data
X, y, feature_names = load_bank_telemarketing_data()
print("Loading Bank Telemarketing Dataset...")
print(f"Loaded Dataset : {len(y):,} samples with {len(feature_names)} features.")
print("\nDataset Characteristics:")
print(f"Total Samples: {len(y):,}")
print(f"Features : {len(feature_names)}")
print(f"No Subscription (0) : {(y == 0).sum():,} ({(y == 0).mean() * 100:.1f}%)")
print(f"Subscription (1): {(y == 1).sum():,} ({(y == 1).mean() * 100:.1f}%)")
print("This represents real scenario of bank telemarketing campaigns.\n")

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Ensure the split data as a numpy array
y_train = np.array(y_train)
y_test = np.array(y_test)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Set : {X_train_scaled.shape[0]:,} samples")
print(f"Test Set : {X_test_scaled.shape[0]:,} samples")
print(f"Training Subscription Cases: {(y_train==1).sum():,} ({(y_train==1).mean() * 100:.1f}%)")
print(f"Test Subscription Cases: {(y_test==1).sum():,} ({(y_test==1).mean() * 100:.1f}%)")


Dataset shape: (41188, 21)
Target Distribution:
No Subscription (0) : 88.7%
Subscription (1): 11.3%
Loading Bank Telemarketing Dataset...
Loaded Dataset : 41,188 samples with 20 features.

Dataset Characteristics:
Total Samples: 41,188
Features : 20
No Subscription (0) : 36,548 (88.7%)
Subscription (1): 4,640 (11.3%)
This represents real scenario of bank telemarketing campaigns.

Training Set : 32,950 samples
Test Set : 8,238 samples
Training Subscription Cases: 3,712 (11.3%)
Test Subscription Cases: 928 (11.3%)


We're working with realistic sceanrio if bank telemarketing dataset from Protuguese telemarketing campaigns - 41188 contacts with 11.3% subscription rate for term deposits. This represent a  business imbalanced dataset reflects real-world challenges in predicting term deposit subscriptions with rare positive cases.

The dataset contains 20 features including customer demographics(age, job, education, personal loan, housing loan, etc.,), campaign details(contact method, duration, previous, etc.) and social-economic indicators. The 7.9:1 class imbalance makes this perfect for testing imbalance handling techniques.

Training : 32,950 samples (3,712 subscriptions)
Testing : 8,238 samples (928 subscriptions)